In [1]:
from typing import TypedDict

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    max_retries=5,    
)

class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str


def generate_joke(state: JokeState):
    prompt = f'Generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content
    return {'joke': response}


def generate_explanation(state: JokeState):
    prompt = f'Write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content
    return {'explanation': response}

In [2]:
checkpointer = InMemorySaver()

graph = StateGraph(JokeState)
graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)
graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

workflow = graph.compile(checkpointer=checkpointer)

In [3]:
config = {'configurable': {'thread_id': '1'}}

workflow.invoke({'topic': 'pizza'}, config=config)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'Why did the pizza go to the psychologist?\n\nBecause it was having a *slice* identity crisis and felt like everyone was just toppings-obsessed!',
   'extras': {'signature': 'EmAKXgFpFH0Taj4lM1Tw+NcGXXjVKoOz/idWpPRkAJq00aoUCWQ63mlnASQ4QdzZSiQdxNA2EJeD7u6IdvBLb3GtX/sF/2JCu+zlVQ/Kv6fq2LrsGnTASabbUkoEC0JBUl4='}}],
 'explanation': [{'type': 'text',
   'text': 'Here is an explanation of the joke:\n\n1. **"Slice" identity crisis:** This is a play on the psychological term **"identity crisis"** (a period of uncertainty and confusion about one\'s identity and role in life). Since the patient is a pizza, its identity crisis revolves around being cut into "slices." \n2. **Toppings-obsessed:** This plays on the feeling of being overlooked for one\'s inner qualities—in a human, this might be personality, but for a pizza, it means people only care about the external toppings rather than the crust, sauce, or the pizza as a whole. \n\nIn short,

In [4]:
workflow.get_state(config)

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the pizza go to the psychologist?\n\nBecause it was having a *slice* identity crisis and felt like everyone was just toppings-obsessed!', 'extras': {'signature': 'EmAKXgFpFH0Taj4lM1Tw+NcGXXjVKoOz/idWpPRkAJq00aoUCWQ63mlnASQ4QdzZSiQdxNA2EJeD7u6IdvBLb3GtX/sF/2JCu+zlVQ/Kv6fq2LrsGnTASabbUkoEC0JBUl4='}}], 'explanation': [{'type': 'text', 'text': 'Here is an explanation of the joke:\n\n1. **"Slice" identity crisis:** This is a play on the psychological term **"identity crisis"** (a period of uncertainty and confusion about one\'s identity and role in life). Since the patient is a pizza, its identity crisis revolves around being cut into "slices." \n2. **Toppings-obsessed:** This plays on the feeling of being overlooked for one\'s inner qualities—in a human, this might be personality, but for a pizza, it means people only care about the external toppings rather than the crust, sauce, or the pizza as a whole. \n\

In [5]:
list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the pizza go to the psychologist?\n\nBecause it was having a *slice* identity crisis and felt like everyone was just toppings-obsessed!', 'extras': {'signature': 'EmAKXgFpFH0Taj4lM1Tw+NcGXXjVKoOz/idWpPRkAJq00aoUCWQ63mlnASQ4QdzZSiQdxNA2EJeD7u6IdvBLb3GtX/sF/2JCu+zlVQ/Kv6fq2LrsGnTASabbUkoEC0JBUl4='}}], 'explanation': [{'type': 'text', 'text': 'Here is an explanation of the joke:\n\n1. **"Slice" identity crisis:** This is a play on the psychological term **"identity crisis"** (a period of uncertainty and confusion about one\'s identity and role in life). Since the patient is a pizza, its identity crisis revolves around being cut into "slices." \n2. **Toppings-obsessed:** This plays on the feeling of being overlooked for one\'s inner qualities—in a human, this might be personality, but for a pizza, it means people only care about the external toppings rather than the crust, sauce, or the pizza as a whole. \n

# Time Travel

In [ ]:
# find the checkpoint where topic='pizza' exists but joke hasn't generated yet,
# copy its checkpoint_id, then:

config = {'configurable': {'thread_id': '1', 'checkpoint_id': '<that-id>'}}

# read the state AT that checkpoint:
workflow.get_state(config)
# {'topic': 'pizza'}   <-- just topic, pre-joke

# replay forward from there:
workflow.invoke(None, config=config)

# Updating State

In [ ]:
# change the topic at a specific checkpoint, then replay:
new_config = workflow.update_state(
    {'configurable': {'thread_id': '1', 'checkpoint_id': '<pizza-topic-checkpoint-id>'}},
    {'topic': 'samosa'}
)

# IMPORTANT: update_state returns a NEW checkpoint_id for the forked branch.
# Replay from THAT returned config, not the original:
workflow.invoke(None, new_config)